# The same simulation, with the clustered variables made CONDITIONS

Identical to `region_space.ipynb` -- same analyses, same code, same figures --
except that a neuron is now a conjunctive 2-D von Mises cell with preferred angles
$(\theta_1, \theta_2)$, all neurons sharing one width and skew. `a` and $\\psi$ now
shape the cloud of *preferred angles*.

Because the angles enter the tuning nonlinearly, the population geometry no longer
depends on only the first two moments of that cloud, so `a` is not guaranteed to be
invisible to it -- which is the point of running this version.

## Categoricality and whether regions form types

A region's neurons are an elongated cloud in tuning-shape space. `amount` sets how bimodal
it is along its long axis -- how categorical the region is. `kind` ($\psi$) sets that axis's
orientation -- what the region is categorical *about*.

**Scenario 1 -- a continuum.** Regions vary continuously: `amount` sweeps from 0 to ~1 and
`kind` sweeps with it. The distribution of per-region categoricality is **spread out**, and
in region space the regions lie on a continuum with no types in it.

**Scenario 2 -- three kinds.** **No region is categorical at all** ($a$ = 0, so every
per-region $z$ sits at zero), but `kind` takes three discrete values, so region space
contains three clean clusters.

Read the first column and you learn nothing about the third.

### Why `amount` and `kind` are swept together in scenario 1

The generator holds a region's total spread fixed as `amount` changes, so the second moment
of its weights -- which is what a Procrustes comparison of regions sees -- depends on `kind`
and not on `amount`. Sweeping `amount` alone would spread the categoricality histogram while
leaving region space almost unchanged; sweeping `kind` alone would give the continuum but a
flat histogram. Scenario 1 needs both, and they are two visible consequences of one
underlying gradient.

### Two choices in scenario 1 that are not arbitrary

Its regions are **sampled** along the gradient rather than evenly spaced: an evenly spaced
lattice is itself non-Gaussian, and the region-space test would report that regularity as
structure.

And it sweeps only part of the $\psi$ range. Procrustes distance is a nonlinear function of
$\psi$, so a wide sweep puts the regions on a strongly **curved** arc -- and the
silhouette-versus-Gaussian test detects departure from a Gaussian, which curvature is. Over
the full range this simulation returns $p$ = 0.01 for a continuum containing no types
whatsoever. **That is a limitation of the test, not evidence of types**, and it is worth
knowing: a strongly curved continuum of regions can be reported as lumpy. Scenario 1 stays
in the near-straight regime so the intended contrast is the only thing on show.

### The region-space test

Regions are embedded by **classical MDS -- PCA of the Procrustes distances** (deterministic,
and its axes come out ordered by variance, unlike SMACOF). The statistic is the best
silhouette over $k$; the null is Gaussians matched to that embedding's mean and covariance.
A continuum is *not* lumpy in this sense -- a line of points and a Gaussian blob score
alike -- while genuine types are.

In [ ]:
from pathlib import Path

from shapemetrics import paths
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, Patch
from scipy import stats

# Anchor on the directory holding the simulation modules, so the notebook runs
# whether the kernel starts in clustering-simulation or at the repo root.
# Path.cwd() alone fails with ModuleNotFoundError from anywhere else.
R = paths.figure_code("Figure1").theta_space

OUT = paths.set_figure("Figure1")
plt.rcParams.update({"svg.fonttype": "none", "text.usetex": False})

# house style; the three-hue set passes the lightness, chroma, CVD (worst dE 22.1)
# and contrast checks, and keeps the two hues used by the other notebooks here
PANEL = 2.1
OBS, NULLC, GREY = "#B22222", "#1F6FB2", "0.75"
KIND_COLORS = ["#B22222", "#1F6FB2", "#E8A33D"]
N_DRAWS = 200
N_DRAWS_CAT = 25   # null draws per region for the categoricality z

print(f"{R.N_REGIONS} regions x {R.N_NEURONS} neurons")

## Running both scenarios

In [ ]:
f = OUT / "theta_3x4.npz"
if f.exists():
    res = np.load(f, allow_pickle=True)["res"].item()
else:
    res = {}
    for name in ("continuum", "no_gradient", "kinds"):
        X, region, colour = R.scenario(name)
        E = R.pca_embed(R.procrustes_distances(X, region))
        res[name] = dict(
            colour=colour, emb=E,
            z=R.categoricality(X, region, np.random.default_rng(0), N_DRAWS_CAT),
            ct=R.continuum_or_types(E, N_DRAWS),
        )
    np.savez(f, res=np.array(res, dtype=object))

for name, r in res.items():
    print(f"\n=== {name} ===")
    print(f"  categoricality z : {r['z'].min():+.2f} to {r['z'].max():+.2f}"
          f"   (mean {r['z'].mean():+.2f} +/- {r['z'].std():.2f})")
    zz = r["z"]
    print(f"  regions with |z| > 1.96 : {int(np.sum(abs(zz) > 1.96))} of {len(zz)}"
          f"   |z| > {stats.norm.ppf(1 - 0.025 / len(zz)):.2f} (corrected):"
          f" {int(np.sum(abs(zz) > stats.norm.ppf(1 - 0.025 / len(zz))))}")
    print(f"  continuum or types: obs {r['ct']['obs']:.3f}  "
          f"null {r['ct']['null'].mean():.3f} +/- {r['ct']['null'].std():.3f}"
          f"   z = {r['ct']['z']:+.2f}   p = {r['ct']['p']:.3f}")

## Is there a gradient along the region cloud?

Rows 1 and 2 give the same categoricality distribution and the same verdict in region space
(one cloud, not types). They differ only in whether categoricality varies *systematically*
along that cloud.

In [ ]:
for name in ("continuum", "no_gradient"):
    r = res[name]
    rr = stats.spearmanr(r["z"], r["emb"][:, 0])
    print(f"{name:>12}: rho(categoricality, region PC1) = {abs(rr.statistic):.2f}"
          f"   p = {rr.pvalue:.1e}")

## The averaging alternative

The same regions, put in a common space a different way: average each region's neurons into
one tuning vector and PCA across regions -- the analysis of Posani et al.'s Fig. 2c. It is
essentially free to compute (no pairwise Procrustes), and it keeps only the first moment.

In [ ]:
emb_mean = {}
for name in ("continuum", "no_gradient", "kinds"):
    X, region, _ = R.scenario(name)
    emb_mean[name] = R.mean_embed(X, region)

rho_mean = stats.spearmanr(res["continuum"]["colour"], emb_mean["continuum"][:, 0])
rho_proc = stats.spearmanr(res["continuum"]["colour"], res["continuum"]["emb"][:, 0])
print(f"continuum: |rho(PC1, position)|   averaged {abs(rho_mean.statistic):.2f}"
      f"  (p = {rho_mean.pvalue:.2f})   vs Procrustes {abs(rho_proc.statistic):.2f}")

c, E = res["kinds"]["colour"], emb_mean["kinds"]
sep = np.mean([np.linalg.norm(E[c == i].mean(0) - E[c == j].mean(0))
               for i in range(R.N_KINDS) for j in range(i + 1, R.N_KINDS)])
within = np.mean([E[c == k].std(0).mean() for k in range(R.N_KINDS)])
verdict = "the kinds overlap" if sep < within else "the kinds are SEPARATED"
print(f"kinds    : centroid separation {sep:.3f} vs within-kind spread {within:.3f}"
      f"   -> {verdict}")

## Example regions in the plane of preferred angles

What the two scenarios look like at the level of a single region. Each point is one neuron,
placed by its preferred $\theta_1$ and $\theta_2$. Nothing new is computed here: each
neuron's preferred angles are simply read back off its own simulated responses.

In [ ]:
# X = MEAN + SCALE * W @ MODES, and MODES is orthonormal, so the neurons'
# shape coordinates come straight back out.  Mode 2 increases with the von Mises
# concentration kappa (r = +0.99), i.e. with NARROWER tuning, so it is negated
# here to give a true width axis (r = +0.98 with the measured circular s.d.).
def shape_coords(X):
    # the analogue of projecting onto the shape modes: each neuron's preferred
    # (theta1, theta2), measured from its own responses
    return R.preferred_angles(X)


# Four example regions generated directly, so that each row varies ONE knob.
# Row 1 holds psi fixed and changes a; row 2 holds a = 0 and changes psi.
PSI_ROW1 = R.PSI                                   # +45 deg, fixed across row 1
KIND_PSI = np.linspace(-R.PSI, R.PSI, R.N_KINDS)   # the three kinds

amounts = [0.0, 0.95, 0.0, 0.0]
psis_ex = [PSI_ROW1, PSI_ROW1, KIND_PSI[0], KIND_PSI[1]]
Xex, regex = R.simulate(amounts, psis_ex, seed=1)
Wex = shape_coords(Xex)

cmap = plt.get_cmap("viridis")
titles = ["$a$ = 0", "$a$ = 0.95", "kind 1", "kind 2"]
cols = [cmap(0.0), cmap(1.0), KIND_COLORS[0], KIND_COLORS[1]]
rowlab = ["only $a$ changes\n(how categorical)", "only $\\psi$ changes\n(what varies)"]

clouds = [Wex[regex == i] - Wex[regex == i].mean(0) for i in range(4)]
lim = 1.12 * np.percentile(np.abs(np.concatenate(clouds)), 99.5)

fig, axes = plt.subplots(2, 2, figsize=(2 * PANEL, 2 * PANEL))
for i, ax in enumerate(axes.ravel()):
    P = clouds[i]
    u = np.array([np.cos(psis_ex[i]), -np.sin(psis_ex[i])])   # y is flipped
    ax.axhline(0, color="0.85", lw=0.8, zorder=0)
    ax.axvline(0, color="0.85", lw=0.8, zorder=0)
    ax.plot([-lim * u[0], lim * u[0]], [-lim * u[1], lim * u[1]],
            color="0.65", lw=1, ls="--", zorder=1)
    ax.scatter(P[:, 0], P[:, 1], s=11, facecolor=cols[i], edgecolor="black",
               linewidths=0.35, alpha=0.9, zorder=2)
    ax.set(xlim=(-lim, lim), ylim=(-lim, lim), xticks=[], yticks=[])
    ax.set_xlabel("skew", fontsize=7.5)
    ax.set_ylabel("width", fontsize=7.5, labelpad=2)
    ax.set_title(titles[i], fontsize=8.5, pad=4)
    ax.set_box_aspect(1)
    sns.despine(ax=ax, left=True, bottom=True)

for row in (0, 1):
    axes[row, 0].text(-0.34, 0.5, rowlab[row], fontsize=8, rotation=90,
                      va="center", ha="center", transform=axes[row, 0].transAxes)

fig.tight_layout(w_pad=1.4, h_pad=2.0)
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"region_examples.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()

## Clustering neurons within a region, versus pooling them across regions

The per-region categoricality is matched across the three fields by construction. So if
clustering the **pooled** neurons gives a different answer in the three, that difference can
only come from how the regions are arranged relative to one another -- structure that has
nothing to do with whether any region contains cell types.

The pooled cloud is subsampled to `N_POOL` neurons: silhouette is $O(n^2)$, and the full
6000 would take over an hour across the three fields and their nulls.

In [ ]:
import shapemetrics as sm

N_POOL, N_NULL_POOL = 2000, 100

f = OUT / "pooled_vs_theta.npz"
if f.exists():
    pooled = np.load(f, allow_pickle=True)["pooled"].item()
else:
    pooled = {}
    for name in ("continuum", "no_gradient", "kinds"):
        X, region, _ = R.scenario(name)
        rng = np.random.default_rng(0)
        Xp = X[rng.choice(len(X), N_POOL, replace=False)]
        obs = sm.pipeline_silhouette(Xp, k_lim=R.KLIM, n_init=R.NINIT)
        nl = np.array([sm.pipeline_silhouette(sm.curve_gaussian_null(Xp, rng),
                                              k_lim=R.KLIM, n_init=R.NINIT)
                       for _ in range(N_NULL_POOL)])
        pooled[name] = dict(obs=obs, null=nl,
                            z=float((obs - nl.mean()) / (nl.std() + 1e-12)),
                            p=float((np.sum(nl >= obs) + 1) / (len(nl) + 1)))
    np.savez(f, pooled=np.array(pooled, dtype=object))

print(f"{'':>12} {'per region (mean z)':>20} {'pooled (z)':>12}")
for name in ("continuum", "no_gradient", "kinds"):
    print(f"{name:>12} {res[name]['z'].mean():>20.2f} {pooled[name]['z']:>12.2f}")

fig, axes = plt.subplots(1, 2, figsize=(2 * PANEL, PANEL))
names = ["continuum", "no_gradient", "kinds"]
lab = ["gradient", "unstructured", "clustered"]
x = np.arange(3)

ax = axes[0]
ax.bar(x, [res[n]["z"].mean() for n in names], color=GREY, width=0.6,
       yerr=[res[n]["z"].std() / np.sqrt(len(res[n]["z"])) for n in names],
       error_kw=dict(lw=1, ecolor="0.35"))
ax.axhline(1.96, color="0.45", lw=1, ls="--")
ax.set(xticks=x, ylabel="categoricality (z)")
ax.set_xticklabels(lab, fontsize=6.5, rotation=20, ha="right")
ax.set_title("neurons clustered\nwithin each region", fontsize=8, pad=3)
ax.set_box_aspect(1); sns.despine(ax=ax)

ax = axes[1]
ax.bar(x, [pooled[n]["z"] for n in names], color=OBS, width=0.6)
ax.axhline(1.96, color="0.45", lw=1, ls="--")
ax.set(xticks=x, ylabel="categoricality (z)")
ax.set_xticklabels(lab, fontsize=6.5, rotation=20, ha="right")
ax.set_title("neurons pooled\nacross regions", fontsize=8, pad=3)
ax.set_box_aspect(1); sns.despine(ax=ax)

fig.tight_layout(w_pad=1.8)
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"pooled_vs_theta.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()

## The figure

In [ ]:
ROWS = [("continuum", res["continuum"], "1: gradient"),
        ("no_gradient", res["no_gradient"], "2: unstructured"),
        ("kinds", res["kinds"], "3: clustered")]
TITLE = {"continuum": "gradient", "no_gradient": "unstructured",
         "kinds": "clustered"}

# shared bins so the categoricality histograms are directly comparable
allz = np.concatenate([r["z"] for _, r, _ in ROWS])
ZBINS = np.linspace(np.floor(allz.min()) - .5, np.ceil(allz.max()) + .5, 22)


def plot_hist(ax, r):
    ax.hist(r["z"], bins=ZBINS, color=GREY)
    ax.axvline(0, color="0.45", lw=1, ls="--")
    ax.set_xlim(ZBINS[0], ZBINS[-1])
    ylo, yhi = ax.get_ylim(); ax.set_ylim(0, yhi * 1.26)
    ax.axvspan(-1.96, 1.96, color="0.89", lw=0, zorder=0)
    ax.text(-1.85, yhi * 1.12, "n.s.", ha="left", va="center", fontsize=6.5,
            color="0.4")
    ax.set(xlabel="categoricality (z)", yticks=[])
    ax.set_box_aspect(1); sns.despine(ax=ax, left=True)


# rows 1-2 are coloured by the RANK of each region's categoricality: z is
# strongly right-skewed, so a linear scale hides the ordering
def plot_regions(ax, E, name, r, legend=False):
    if name == "kinds":
        for k in range(R.N_KINDS):
            m = r["colour"] == k
            ax.scatter(E[m, 0], E[m, 1], c=KIND_COLORS[k], s=22,
                       edgecolor="black", linewidths=0.3, label=f"kind {k + 1}")
        if legend:
            ax.legend(frameon=False, fontsize=5.6, loc="best", handlelength=1.0,
                      scatterpoints=1, labelspacing=0.25)
    else:
        ax.scatter(E[:, 0], E[:, 1], c=stats.rankdata(r["z"]) / len(r["z"]),
                   cmap="viridis", s=22, edgecolor="black", linewidths=0.3,
                   vmin=0, vmax=1)
    ax.set(xlabel="region PC 1", ylabel="region PC 2", xticks=[], yticks=[])
    ax.set_box_aspect(1); sns.despine(ax=ax, left=True, bottom=True)


# the three silhouette histograms share bins AND range, so the panels are
# directly comparable: the observed value for the three kinds sits far to the
# right of where the other two fall
_v = np.concatenate([np.r_[r["ct"]["null"], r["ct"]["obs"]] for _, r, _ in ROWS])
BINW = min(np.ptp(r["ct"]["null"]) for _, r, _ in ROWS) / 22
SBINS = np.arange(np.floor(_v.min() / BINW) * BINW, _v.max() + 2 * BINW, BINW)


def plot_test(ax, r):
    ct = r["ct"]
    ax.hist(ct["null"], bins=SBINS, color=NULLC, alpha=0.8, label="one blob")
    ax.set_xlim(SBINS[0], SBINS[-1])
    ax.axvline(ct["obs"], color=OBS, lw=2, label=f"regions\n$p$ = {ct['p']:.3f}")
    ax.set(xlabel="best silhouette\nbetween regions", yticks=[])
    ylo, yhi = ax.get_ylim(); ax.set_ylim(ylo, yhi * 1.42)
    ax.legend(frameon=False, fontsize=5.8, loc="upper left", handlelength=1.0,
              labelspacing=0.25)
    ax.set_box_aspect(1); sns.despine(ax=ax, left=True)


# ============ FIGURE 1: what you can see at the neuron level ==============
# the pooled panels share bins and range, as the region-space tests do
_pv = np.concatenate([np.r_[pooled[n]["null"], pooled[n]["obs"]]
                      for n, _, _ in ROWS[:2]])
PBINW = min(np.ptp(pooled[n]["null"]) for n, _, _ in ROWS[:2]) / 18
PBINS = np.arange(np.floor(_pv.min() / PBINW) * PBINW, _pv.max() + 2 * PBINW, PBINW)

fig, axes = plt.subplots(2, 3, figsize=(3 * PANEL, 2 * PANEL))
for row, (name, r, lab) in enumerate(ROWS[:2]):
    plot_hist(axes[row, 0], r)
    axes[row, 0].set_ylabel(lab, fontsize=8, labelpad=6)
    plot_regions(axes[row, 1], emb_mean[name], name, r)
    axes[row, 1].text(0.5, -0.30, "no structure", fontsize=7, color="0.25",
                      ha="center", transform=axes[row, 1].transAxes)

    ax = axes[row, 2]
    pl = pooled[name]
    ax.hist(pl["null"], bins=PBINS, color=NULLC, alpha=0.8, label="one blob")
    ax.axvline(pl["obs"], color=OBS, lw=2,
               label=f"pooled neurons\n$p$ = {pl['p']:.3f}")
    ax.set_xlim(PBINS[0], PBINS[-1])
    # the range is narrow, so the default tick labels run into each other
    ax.xaxis.set_major_locator(plt.MaxNLocator(3))
    ax.tick_params(axis="x", labelsize=7)
    ax.set(xlabel="mean silhouette\n(neurons pooled)", yticks=[])
    ylo, yhi = ax.get_ylim(); ax.set_ylim(ylo, yhi * 1.42)
    ax.legend(frameon=False, fontsize=5.8, loc="upper left", handlelength=1.0,
              labelspacing=0.25)
    ax.set_box_aspect(1); sns.despine(ax=ax, left=True)

    if row == 0:
        axes[row, 0].set_title("how categorical\nis each region?", fontsize=8, pad=3)
        axes[row, 1].set_title("regions, by mean\ntuning curve", fontsize=8, pad=3)
        axes[row, 2].set_title("neurons pooled\nacross regions", fontsize=8, pad=3)
fig.tight_layout(w_pad=1.6, h_pad=2.6)
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"theta_neuron_level.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()

# ============ FIGURE 2: the same three fields in region space =============
fig, axes = plt.subplots(2, 3, figsize=(3 * PANEL, 2 * PANEL))
for col, (name, r, _) in enumerate(ROWS):
    plot_regions(axes[0, col], r["emb"], name, r, legend=(name == "kinds"))
    axes[0, col].set_title(TITLE[name], fontsize=8.5, pad=4)
    rho = abs(stats.spearmanr(r["z"], r["emb"][:, 0]).statistic)
    axes[0, col].text(0.5, -0.30,
                      "three clusters" if name == "kinds"
                      else f"one cloud, $\\rho$ = {rho:.2f}",
                      fontsize=7, color="0.25", ha="center",
                      transform=axes[0, col].transAxes)
    plot_test(axes[1, col], r)
axes[0, 0].set_ylabel("regions, by\nProcrustes distance", fontsize=8, labelpad=6)
axes[1, 0].set_ylabel("continuum,\nor types?", fontsize=8, labelpad=6)
fig.tight_layout(w_pad=1.6, h_pad=3.0)
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"theta_regions.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()

## The combined figure

All of the above in one 2 x 8 panel: the two knobs (columns 1-2), what the neuron level
shows for a gradient and for a random arrangement (columns 3-5), and the same three fields
in region space (columns 6-8).

In [ ]:
PC = 1.7                                    # smaller panels: eight columns
fig, axes = plt.subplots(2, 8, figsize=(8 * PC, 2.15 * PC))

# ---- columns 1-2: example regions in shape space -------------------------
PSI_R1 = R.PSI
KP = np.linspace(-R.PSI, R.PSI, R.N_KINDS)
ex_a, ex_psi = [0.0, 0.95, 0.0, 0.0], [PSI_R1, PSI_R1, KP[0], KP[1]]
Xe, rege = R.simulate(ex_a, ex_psi, seed=1)
We = shape_coords(Xe)
cl = [We[rege == i] - We[rege == i].mean(0) for i in range(4)]
elim = 1.12 * np.percentile(np.abs(np.concatenate(cl)), 99.5)
etit = ["$a$ = 0", "$a$ = 0.95", "", ""]
ecol = ["0.6"] * 4          # single neurons: grey everywhere in this block

# a 2-s.d. Gaussian ellipse per cluster: one component everywhere except
# a = 0.95, where the two lobes are fitted separately
def gauss_ellipse(ax, P, nsd=2.0):
    w, V = np.linalg.eigh(np.cov(P, rowvar=False))
    ax.add_patch(Ellipse(P.mean(0), 2 * nsd * np.sqrt(w[-1]),
                         2 * nsd * np.sqrt(w[-2]),
                         angle=np.degrees(np.arctan2(V[1, -1], V[0, -1])),
                         fill=False, lw=0.9, ec="black", zorder=3))


for n, (rr, cc) in enumerate([(0, 0), (0, 1), (1, 0), (1, 1)]):
    ax = axes[rr, cc]
    u = np.array([np.cos(ex_psi[n]), -np.sin(ex_psi[n])])
    ax.axhline(0, color="0.85", lw=0.7, zorder=0)
    ax.axvline(0, color="0.85", lw=0.7, zorder=0)
    ax.plot([-elim * u[0], elim * u[0]], [-elim * u[1], elim * u[1]],
            color="0.65", lw=0.9, ls="--", zorder=1)
    ax.scatter(cl[n][:, 0], cl[n][:, 1], s=6, facecolor=ecol[n],
               edgecolor="black", linewidths=0.25, alpha=0.9, zorder=2)
    if n == 1:                                  # a = 0.95: two lobes
        proj = cl[n] @ u
        for m in (proj < 0, proj >= 0):
            gauss_ellipse(ax, cl[n][m])
    else:
        gauss_ellipse(ax, cl[n])
    ax.set(xlim=(-elim, elim), ylim=(-elim, elim), xticks=[], yticks=[])
    ax.set_xlabel("pref. $\\theta_1$", fontsize=6.5)
    ax.set_ylabel("pref. $\\theta_2$", fontsize=6.5, labelpad=1)
    ax.set_title(etit[n], fontsize=7.5, pad=3)
    ax.set_box_aspect(1); sns.despine(ax=ax, left=True, bottom=True)

# ---- columns 3-5: the neuron level, for scenarios 1 and 2 ----------------
for row, (name, r, lab) in enumerate(ROWS[:2]):
    plot_hist(axes[row, 2], r)
    axes[row, 2].set_ylabel(lab, fontsize=7.5, labelpad=4)
    axes[row, 2].set_xlabel("categoricality (z)", fontsize=6.5)
    plot_regions(axes[row, 4], emb_mean[name], name, r)
    ax = axes[row, 3]
    pl = pooled[name]
    ax.hist(pl["null"], bins=PBINS, color=NULLC, alpha=0.8)
    ax.axvline(pl["obs"], color=OBS, lw=1.8)
    ax.set_xlim(PBINS[0], PBINS[-1])
    ax.xaxis.set_major_locator(plt.MaxNLocator(3))
    ax.tick_params(axis="x", labelsize=6)
    ax.set(yticks=[]); ax.set_xlabel("pooled silhouette", fontsize=6.5)
    ylo, yhi = ax.get_ylim(); ax.set_ylim(ylo, yhi * 1.45)
    ax.legend([Patch(facecolor=NULLC), Line2D([0], [0], color=OBS, lw=1.8)],
              ["null", "data"], frameon=False, fontsize=5.5, loc="upper left",
              handlelength=0.9, labelspacing=0.2, borderaxespad=0.1)
    ax.set_box_aspect(1); sns.despine(ax=ax, left=True)
axes[0, 2].set_title("is each region\ncategorical?", fontsize=7.5, pad=3)
axes[0, 3].set_title("neurons pooled\nacross regions", fontsize=7.5, pad=3)
axes[0, 4].set_title("regions, by mean\ntuning curve", fontsize=7.5, pad=3)

# ---- columns 6-8: the same three fields in region space ------------------
for col, (name, r, _) in enumerate(ROWS):
    a0, a1 = axes[0, 5 + col], axes[1, 5 + col]
    plot_regions(a0, r["emb"], name, r)
    a0.set_title(TITLE[name], fontsize=7.5, pad=3)
    plot_test(a1, r)
    a1.set_xlabel("best silhouette", fontsize=6.5)
    a1.legend([Patch(facecolor=NULLC), Line2D([0], [0], color=OBS, lw=1.8)],
              ["null", "data"], frameon=False, fontsize=5.5, loc="upper left",
              handlelength=0.9, labelspacing=0.2, borderaxespad=0.1)

for ax in axes.ravel():
    ax.xaxis.label.set_size(6.5); ax.yaxis.label.set_size(6.5)

fig.tight_layout(w_pad=0.9, h_pad=2.2)
fig.canvas.draw()
for x0, x1, txt in ((2, 4, "neuron level"), (5, 7, "region space")):
    b0, b1 = axes[0, x0].get_position(), axes[0, x1].get_position()
    fig.text((b0.x0 + b1.x1) / 2, 1.03, txt, fontsize=9, ha="center", va="bottom")
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"theta_full.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()

## Reading the figure

The dissociation survives the change of generator, and three things about it change.

**The three fields still share a categoricality distribution and still differ in region
space.** Mean per-region $z$ is +6.25, +6.14 and +5.92 -- statistically the same histogram
three times over -- while the region-space verdict is one continuous cloud for the gradient
($p$ = 0.114) and for the unstructured field ($p$ = 0.219), and three clean types for the
clustered field ($z$ = +13.0, $p$ = 0.005). Region PC1 still recovers position along the
gradient at $|\rho|$ = 0.98. That is the same conclusion the shape-space version reaches, so
it does not depend on the special covariance-preserving construction used there.

**Categoricality now saturates.** In shape space only ~20 of 60 regions per field cleared
$|z|$ = 1.96 and the range dipped negative; here **all 60 of 60** clear even the
60-test-corrected threshold, in every field, with a floor of $z$ = +3.4. Two neurons with
different preferred angles are far apart in condition space in a way two neurons with
different tuning *shapes* are not, so the same values of $a$ produce far more separable
clusters. The histogram is still matched across fields, so the figure's argument holds -- but
the measure is at ceiling and can no longer rank regions: it tracks region PC1 at
$\rho$ = 0.09 along the gradient, where in shape space it tracked it at 0.72.

**The cheap averaging alternative stops being a foil.** This is the real casualty. In shape
space, collapsing each region to its mean tuning vector destroyed the structure -- it
recovered the gradient at $|\rho|$ = 0.04 and left the three kinds overlapping (separation
0.038 vs spread 0.143). Here it recovers the gradient at $|\rho|$ = 0.95 and separates the
kinds cleanly (5.627 vs 0.420). Preferred angles show up in a region's *first* moment, so
averaging keeps them; tuning shapes did not, so averaging lost them. Whether the population
geometry buys anything over a per-region average is therefore a property of what the neurons
vary in, not a general fact.

## What to take from it

The left column and the right column answer different questions and neither constrains the
other -- a field of uniformly categorical regions forms a seamless continuum in one scenario
and three sharp types in another, with the same categoricality histogram.

What the two notebooks together show is that the *strength* of the argument depends on the
parameterisation. When the clustered variables enter the tuning linearly (shape space) the
geometry is a function of the cloud's first two moments alone, so `a` is invisible to it
**exactly** and the mean embedding is a genuinely weaker instrument. When they enter
nonlinearly (here), higher moments leak into the geometry -- an isolated field varying only
in $a$ gives $|\rho|$(region PC1, $a$) = 0.63 rather than 0.00 -- and the first moment
already carries most of what there is to see. The dissociation still comes out, but it is
driven by $\psi$ rather than protected by an exact orthogonality.

**The caveat on the region-space test carries over unchanged.** It compares the best
silhouette against a Gaussian, so it detects departure from a Gaussian, not clustering
specifically; a strongly curved continuum is such a departure. That was established in
`region_space.ipynb` and is a property of the test, not of the generator.

## A third way to put regions in a common space: the categoricality scalar

Describe each region by its categoricality $z$ alone and set $D_{ij} = |z_i - z_j|$, then
embed as before. This costs nothing -- $z$ is already in hand -- and it does put every region
in one space. But a single number per region can span only **one** dimension, so the second
axis is exactly zero: whatever regions differ by other than "how categorical am I" is
invisible here by construction.

In [ ]:
emb_z = {name: R.pca_embed(np.abs(r["z"][:, None] - r["z"][None, :]), 2)
         for name, r in res.items()}

for name, E in emb_z.items():
    print(f"{name:>10}: axis 1 spread {E[:, 0].std():.3f}   "
          f"axis 2 spread {E[:, 1].std():.1e}  (zero by construction)")

rz = stats.spearmanr(res["continuum"]["colour"], emb_z["continuum"][:, 0])
rp = stats.spearmanr(res["continuum"]["colour"], res["continuum"]["emb"][:, 0])
print(f"\ncontinuum: |rho(axis 1, position)|   categoricality {abs(rz.statistic):.2f}"
      f"   vs Procrustes {abs(rp.statistic):.2f}")

c, E = res["kinds"]["colour"], emb_z["kinds"]
sep = np.mean([abs(E[c == i, 0].mean() - E[c == j, 0].mean())
               for i in range(R.N_KINDS) for j in range(i + 1, R.N_KINDS)])
within = np.mean([E[c == k, 0].std() for k in range(R.N_KINDS)])
verdict = "the kinds overlap" if sep < within else "the kinds are SEPARATED"
print(f"kinds    : centroid separation {sep:.3f} vs within-kind spread {within:.3f}"
      f"   -> {verdict}")

fig, axes = plt.subplots(1, 2, figsize=(2 * PANEL, PANEL))
for ax, (name, r) in zip(axes, res.items()):
    E = emb_z[name]
    if name == "continuum":
        ax.scatter(E[:, 0], E[:, 1], c=r["colour"], cmap="viridis", s=26,
                   edgecolor="black", linewidths=0.35)
        ax.set_title("a continuum", fontsize=8, pad=3)
    else:
        for k in range(R.N_KINDS):
            m = r["colour"] == k
            ax.scatter(E[m, 0], E[m, 1], c=KIND_COLORS[k], s=26,
                       edgecolor="black", linewidths=0.35, label=f"kind {k + 1}")
        ax.legend(frameon=False, fontsize=6, loc="upper right", handlelength=1.0,
                  scatterpoints=1, labelspacing=0.3)
        ax.set_title("three kinds", fontsize=8, pad=3)
    # axis 2 is numerically zero, so fix the y range rather than let it
    # autoscale on noise: the flat line IS the result
    span = E[:, 0].max() - E[:, 0].min()
    ax.set(xticks=[], yticks=[], ylim=(-0.35 * span, 0.35 * span))
    ax.set_xlabel("region axis 1", fontsize=7.5)
    ax.set_ylabel("region axis 2", fontsize=7.5, labelpad=2)
    ax.set_box_aspect(1)
    sns.despine(ax=ax, left=True, bottom=True)

fig.suptitle("regions by categoricality alone:  $D_{ij} = |z_i - z_j|$", fontsize=8.5)
fig.tight_layout(w_pad=1.6)
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"theta_z_embedding.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()

## The example regions as population manifolds

The same four example regions, seen the other way round. Each region's activity is a
$256 \times 100$ matrix (the $16 \times 16$ grid of $(\theta_1, \theta_2)$ by neurons);
projecting those $256$ conditions onto
that region's own first three principal components traces its population manifold -- the
object a Procrustes distance actually compares.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)
from sklearn.decomposition import PCA

# the four example regions of the combined figure
PSI_R1 = R.PSI
KP = np.linspace(-R.PSI, R.PSI, R.N_KINDS)
ex_a, ex_psi = [0.0, 0.95, 0.0, 0.0], [PSI_R1, PSI_R1, KP[0], KP[1]]
etit = ["$a$ = 0", "$a$ = 0.95", "kind 1", "kind 2"]
Xe, rege = R.simulate(ex_a, ex_psi, seed=1)

fig = plt.figure(figsize=(4 * PANEL, 1.15 * PANEL))
for i in range(4):
    Xi = Xe[rege == i]                       # neurons x theta-bins
    # conditions as samples: the manifold traced out as (theta1, theta2) vary.
    # the conditions are a 2-D grid, not a 1-D sweep, so the sheet is drawn as a
    # wireframe over the grid rather than as a single connecting line
    M = PCA(3).fit_transform(Xi.T - Xi.T.mean(0))
    G = M.reshape(R.NG, R.NG, 3)
    ax = fig.add_subplot(1, 4, i + 1, projection="3d")
    for L in (G, G.transpose(1, 0, 2)):
        for row in L:
            ax.plot(row[:, 0], row[:, 1], row[:, 2], color="0.75", lw=0.4, zorder=1)
    ax.scatter(M[:, 0], M[:, 1], M[:, 2], c=np.degrees(R.TH1), cmap="hsv",
               s=9, edgecolor="black", linewidths=0.2, zorder=2)
    ax.set_title(etit[i], fontsize=8, pad=0)
    for a_, lab in ((ax.xaxis, "PC 1"), (ax.yaxis, "PC 2"), (ax.zaxis, "PC 3")):
        a_.set_ticklabels([]); a_.set_tick_params(length=0)
    ax.set_xlabel("PC 1", fontsize=6, labelpad=-12)
    ax.set_ylabel("PC 2", fontsize=6, labelpad=-12)
    ax.set_zlabel("PC 3", fontsize=6, labelpad=-12)
    ax.grid(False)
    ax.view_init(elev=22, azim=35)

fig.tight_layout(w_pad=0.5)
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"theta_manifolds.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()